# NSE Reliance Copilot: Prompt-Engineered Equity Research Assistant
**Course-end project | Prompt Engineering**

This notebook implements the prompt layer for an educational equity research copilot focused on Reliance Industries Limited (RIL): prompt iteration, evidence/data integrity, market-event classification, self-consistency, bull/base/bear scenario analysis, multi-turn analyst conversation, LangChain templates, Jinja2 notes, f-string alerts, settings selection, and evaluation.

**Data integrity warning:** synthetic event/QA corpora and synthetic prices are included for reproducible demonstration. Selected FY2024-25 segment facts are transcribed from Reliance's official annual-report website. Replace synthetic prices with the specified historical dataset and verify audited ratios/page-level report evidence before rating publication.

**Compliance:** Educational research support only; not investment advice.

## 1. Setup

In [ ]:
from pathlib import Path
import os, json
import numpy as np
import pandas as pd
from collections import Counter
from jinja2 import Template
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT / "reliance_equity_research_copilot" / "data").exists():
    ROOT = ROOT / "reliance_equity_research_copilot"
DATA, OUT = ROOT / "data", ROOT / "outputs"
OUT.mkdir(exist_ok=True)
COMPLIANCE = "Educational research support only; not investment advice. Verify all facts and consult a qualified adviser."

## 2. Source register and evidence pack

Official sources: [RIL FY2024-25 annual report](https://www.ril.com/ar2024-25/index.html), [official segment overview](https://www.ril.com/ar2024-25/reliance-at-a-glance.html), [NSE annual-report archive](https://www.nseindia.com/companies-listing/corporate-filings-annual-reports). Historical price dataset named in the brief: [Kaggle Reliance dataset](https://www.kaggle.com/datasets/dhruvanurag20/reliance-data).

The table below contains selected official segment facts. Add the report's exact page/section references for Chairman's statement, segment performance, risks, and debt/capex before final submission.

In [ ]:
segments = pd.read_csv(DATA / "verified_ril_fy2025_segment_facts.csv")
display(segments)
evidence_checklist = pd.DataFrame([
 ["Chairman's statement","Extract claims with page/section citations","Pending page-level audit"],
 ["Segment performance","Revenue, EBITDA and growth","Selected facts included; verify in report"],
 ["Risk factors","Extract risks and page references","Pending page-level audit"],
 ["Debt and capex","Debt, cash flow, capex plans and funding","Pending page-level audit"],
], columns=["section","required_evidence","status"])
display(evidence_checklist)

## 3. Price statistics (demo data only)

The calculation code is real, but the included observations are synthetic. Never describe these statistics as Reliance's actual historical performance.

In [ ]:
prices = pd.read_csv(DATA / "synthetic_reliance_prices_demo.csv", parse_dates=["Date"]).sort_values("Date")
prices["daily_return"] = prices["Close"].pct_change()
years = (prices["Date"].iloc[-1] - prices["Date"].iloc[0]).days / 365.25
cagr = (prices["Close"].iloc[-1] / prices["Close"].iloc[0]) ** (1/years) - 1
prices["drawdown"] = prices["Close"] / prices["Close"].cummax() - 1
price_stats = {
 "data_flag":"SYNTHETIC DEMO — replace before research use",
 "start_date":str(prices.Date.min().date()), "end_date":str(prices.Date.max().date()),
 "CAGR_pct":round(cagr*100,2),
 "max_drawdown_pct":round(prices.drawdown.min()*100,2),
 "annualised_volatility_pct":round(prices.daily_return.std()*np.sqrt(252)*100,2),
 "mean_daily_volume":int(prices.Volume.mean())
}
price_stats

## 4. Annotated prompt library and three iterations

**Baseline:** summarize the annual-report excerpt.

**Iteration 1 — scope:** use only supplied evidence and prohibit unsupported assumptions; this reduces hallucination risk.

**Iteration 2 — structure:** require strategy, segment performance, risks, debt/capex, and missing evidence; this improves coverage.

**Iteration 3 — auditability:** require page/section citations or `NOT PROVIDED`, and separate fact from interpretation; this makes claims reviewable.

Every prompt below is labelled with Instruction, Context, Input Data, and Output Indicator.

In [ ]:
PROMPT_LIBRARY = {
 "annual_report_summary_v3": {
  "Instruction":"Summarize only the supplied annual-report excerpt. Separate reported facts from interpretation; cite page/section or write NOT PROVIDED.",
  "Context":"Educational RIL long-horizon research support, not investment advice.",
  "Input Data":"{report_excerpt}; source metadata: {source_metadata}",
  "Output Indicator":"Bullets: strategy/Chairman, segment performance, risks, debt/capex, missing evidence."
 },
 "event_zero_shot": {
  "Instruction":"Classify event type and sentiment; do not infer cause from price moves alone.",
  "Context":"RIL market-event triage; snippets may be synthetic, incomplete, or ambiguous.",
  "Input Data":"{headline}",
  "Output Indicator":"JSON: event_type, sentiment, confidence, evidence_phrase, needs_analyst_review."
 },
 "fundamental_health": {
  "Instruction":"Assess five supplied ratios; do not calculate missing inputs or mix reporting bases.",
  "Context":"Educational fundamental health triage.",
  "Input Data":"{ratios}; annual-report evidence: {evidence}",
  "Output Indicator":"JSON: Green/Amber/Red, supporting/counter evidence, missing inputs, confidence, review flag."
 },
 "scenario_tree": {
  "Instruction":"Compare bull/base/bear over 3–5 years; state assumptions, evidence, risks, monitorables, and pruning criteria.",
  "Context":"Long-horizon RIL research; synthetic prices cannot be treated as real evidence.",
  "Input Data":"{evidence_pack}; {financial_ratios}; {price_statistics}; {risk_register}",
  "Output Indicator":"Scenario table, candidate rating, conviction, rationale, risks, monitorables, evidence gaps, review gate."
 },
 "analyst_conversation": {
  "Instruction":"Defend only when evidence supports the rating; revise or escalate when challenged with stronger evidence.",
  "Context":"{company_context}; rating={rating}; assumptions={assumptions}",
  "Input Data":"Analyst message: {analyst_message}",
  "Output Indicator":"Position: defend/revise/escalate; evidence; changed assumptions; verification step."
 }
}
for name, spec in PROMPT_LIBRARY.items():
    print("\n"+name)
    for k,v in spec.items(): print(f"{k}: {v}")

## 5. LangChain prompt templates

In [ ]:
try:
    from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
    langchain_available = True
except ImportError:
    langchain_available = False
    print("Install langchain-core to run these template examples.")
if langchain_available:
    report_prompt = PromptTemplate.from_template(
      "Instruction: {instruction}\nContext: {context}\nInput Data: {report_excerpt}\nOutput Indicator: {output_indicator}")
    event_prompt = ChatPromptTemplate.from_messages([
      ("system","Classify RIL events. Sentiment: Positive, Negative, Neutral, Ambiguous. Include compliance disclaimer: "+COMPLIANCE),
      ("human","Classify: {headline}. Return JSON.")
    ])
    print(report_prompt.format(instruction=PROMPT_LIBRARY["annual_report_summary_v3"]["Instruction"],
      context=PROMPT_LIBRARY["annual_report_summary_v3"]["Context"],
      report_excerpt="[Insert audited excerpt]",output_indicator=PROMPT_LIBRARY["annual_report_summary_v3"]["Output Indicator"]))

## 6. Event classification: zero-shot, failure analysis, few-shot

In [ ]:
events = pd.read_csv(DATA / "synthetic_market_events_75.csv")
def zero_shot(text):
    t=text.lower()
    if any(x in t for x in ["downgrades","cuts ","cost overruns","outage","declines","weakens","delayed"]): sentiment="Negative"
    elif any(x in t for x in ["raises","upgrades","higher dividend","rises year on year","improves","strengthens","outperform"]): sentiment="Positive"
    elif "reiterates" in t or "same dividend" in t: sentiment="Neutral"
    else: sentiment="Ambiguous"
    rules=[
      ("Guidance",["guidance","outlook"]),("Analyst downgrade",["downgrades"]),
      ("Analyst upgrade",["upgrades"]),("Merger / acquisition",["acquisition","transaction"]),
      ("Capital expenditure",["capex","commissioning","project"]),("Earnings",["ebitda","quarterly","profit","revenue"]),
      ("Debt / liquidity",["net debt","cash flow"]),("Regulatory",["regulator","regulatory"]),
      ("Operations",["plant","utilisation","outage"]),("Market / price",["shares fall","shares outperform"]),
      ("Dividend / capital return",["dividend"]),("ESG / sustainability",["renewable-energy","sustainability"])
    ]
    et=next((label for label,keys in rules if any(k in t for k in keys)),"Other")
    return et,sentiment
pred=events.headline_or_snippet.apply(zero_shot)
events["predicted_event_type"]=[x[0] for x in pred]
events["predicted_sentiment"]=[x[1] for x in pred]
print("Synthetic-corpus type accuracy:",round((events.event_type==events.predicted_event_type).mean(),3))
print("Synthetic-corpus sentiment accuracy:",round((events.sentiment==events.predicted_sentiment).mean(),3))
display(events[["headline_or_snippet","event_type","predicted_event_type","sentiment","predicted_sentiment"]].head(15))

### Three zero-shot failure modes
1. **Mixed signals:** revenue rises but EBITDA declines; a single sentiment can oversimplify.
2. **Regulatory uncertainty:** an inquiry is not a finding of wrongdoing.
3. **Causal overreach:** a price drop alone does not identify its cause.

Few-shot exemplars:
- “Revenue grows but EBITDA falls” → Earnings / Ambiguous.
- “A regulator opens an inquiry; no finding announced” → Regulatory / Ambiguous.
- “Shares fall while the market is flat; cause unknown” → Market / price / Ambiguous.
- “Acquisition terms and valuation not disclosed” → Merger / acquisition / Ambiguous.

Few-shot prompting improves label consistency but remains sensitive to exemplar coverage, wording, and domain shifts.

In [ ]:
FEW_SHOT = [
 {"text":"Revenue grows but EBITDA falls.","event_type":"Earnings","sentiment":"Ambiguous"},
 {"text":"A regulator opens an inquiry; no finding announced.","event_type":"Regulatory","sentiment":"Ambiguous"},
 {"text":"Shares fall while the market is flat; cause unknown.","event_type":"Market / price","sentiment":"Ambiguous"},
 {"text":"Acquisition valuation and terms are not disclosed.","event_type":"Merger / acquisition","sentiment":"Ambiguous"}
]
few_shot_prompt = "Classify using examples:\n" + "\n".join(
 f"- {e['text']} => {e['event_type']} / {e['sentiment']}" for e in FEW_SHOT
) + "\nHeadline: {headline}\nReturn JSON with confidence and analyst_review."
print(few_shot_prompt)

## 7. Five financial ratios and fundamental health

The project pack does not contain all audited balance-sheet/income-statement inputs, so values are intentionally blank. Fill them from the annual report using a consistent consolidated/standalone basis; do not invent figures.

In [ ]:
inputs = {
 "total_debt":None, "total_equity":None,
 "segment_revenue_current":None, "segment_revenue_prior":None,
 "ebitda_current":None, "revenue_current":None,
 "ebit_current":None, "interest_expense":None, "capital_expenditure":None
}
def div(a,b): return None if a is None or b in (None,0) else a/b
ratios = {
 "debt_to_equity":div(inputs["total_debt"],inputs["total_equity"]),
 "segment_revenue_growth":(inputs["segment_revenue_current"]/inputs["segment_revenue_prior"]-1)
   if inputs["segment_revenue_current"] is not None and inputs["segment_revenue_prior"] not in (None,0) else None,
 "EBITDA_margin":div(inputs["ebitda_current"],inputs["revenue_current"]),
 "interest_coverage":div(inputs["ebit_current"],inputs["interest_expense"]),
 "capex_intensity":div(inputs["capital_expenditure"],inputs["revenue_current"])
}
glossary={
 "debt_to_equity":"Total debt / total equity; leverage indicator.",
 "segment_revenue_growth":"Current segment revenue / prior-period segment revenue - 1.",
 "EBITDA_margin":"EBITDA / revenue; operating profitability measure.",
 "interest_coverage":"EBIT / interest expense; ability to cover interest.",
 "capex_intensity":"Capital expenditure / revenue; investment intensity, not a standalone quality measure."
}
display(pd.DataFrame({"ratio":ratios.keys(),"value":ratios.values(),"definition":[glossary[k] for k in ratios]}))
def health_flag(ratios,evidence_complete=False):
    missing=[k for k,v in ratios.items() if v is None]
    if missing or not evidence_complete:
        return {"flag":"Amber","confidence":"Low","missing_inputs":missing,"analyst_review_required":True,
                "note":"Provisional: audited ratio inputs or evidence references are missing."}
    score=0
    if ratios["debt_to_equity"] is not None and ratios["debt_to_equity"]>1.5: score-=1
    if ratios["interest_coverage"] is not None and ratios["interest_coverage"]<2: score-=1
    if ratios["EBITDA_margin"] is not None and ratios["EBITDA_margin"]<0.10: score-=1
    if ratios["segment_revenue_growth"] is not None and ratios["segment_revenue_growth"]<0: score-=1
    if ratios["capex_intensity"] is not None and ratios["capex_intensity"]>0.20: score-=1
    return {"flag":"Red" if score<=-3 else "Amber" if score<=-1 else "Green",
            "confidence":"Medium","missing_inputs":[],"analyst_review_required":False,
            "note":"Illustrative thresholds require analyst validation."}
health_flag(ratios)

### Self-consistency — five assessments

A majority vote helps detect instability but does not turn missing evidence into certainty. Split votes or low confidence go to an analyst.

In [ ]:
votes=["Amber","Amber","Amber","Red","Amber"]
counts=Counter(votes)
majority,n=counts.most_common(1)[0]
{"votes":votes,"counts":dict(counts),"majority":majority,
 "agreement_pct":n/len(votes)*100,"analyst_review_required":len(counts)>1 or n<4}

## 8. Tree-of-thoughts: bull, base, bear

Prune branches when key assumptions are unsupported or when observable indicators contradict the scenario. This is scenario comparison, not a prediction.

In [ ]:
scenarios = [
 {"branch":"Bull","assumptions":"Retail/digital growth persists; O2C margins recover; capex converts into productive capacity.",
  "evidence_needed":"Verified segment trends, cash conversion, capex milestones, debt trend.",
  "risks":"Execution delays, competition, energy-margin weakness, regulation.",
  "monitorables":"Digital services growth, retail EBITDA, O2C EBITDA, net debt, capex delivery.",
  "prune_if":"Core growth engines miss targets for consecutive periods."},
 {"branch":"Base","assumptions":"Consumer businesses grow; energy earnings remain cyclical; investment is funded manageably.",
  "evidence_needed":"Segment results, consolidated cash flow, debt/capex disclosures, actual price history.",
  "risks":"Mixed margins, delayed capex returns, competition.",
  "monitorables":"Segment margins, operating cash flow, net debt, project milestones, return on capital.",
  "prune_if":"Cash flow weakens while leverage rises and project milestones slip."},
 {"branch":"Bear","assumptions":"O2C weakness persists; capex returns disappoint; debt rises or consumer growth slows.",
  "evidence_needed":"Falling EBITDA, weaker cash conversion, project delays, interest burden.",
  "risks":"Energy-cycle downturn, cost overruns, regulation, execution.",
  "monitorables":"EBITDA, interest coverage, debt/equity, capex intensity, segment growth.",
  "prune_if":"Margins recover sustainably and leverage declines with verified cash generation."}
]
display(pd.DataFrame(scenarios))
def rating_gate(health,evidence_complete,price_data_real,votes_agree,conviction):
    if not evidence_complete or not price_data_real or health.get("analyst_review_required",True) or not votes_agree or conviction=="Low":
        return {"candidate_rating":"HOLD (provisional; do not publish)","conviction":"Low",
                "publishable":False,"analyst_review_required":True,
                "reason":"Evidence, price provenance, vote stability, or conviction gate not met."}
    return {"candidate_rating":"Human analyst decision required","publishable":False,"analyst_review_required":True}
rating_gate(health_flag(ratios),False,False,False,"Low")

## 9. Chain-of-thought vs tree-of-thoughts comparison memo

Use structured stepwise analysis for bounded checks such as interpreting five ratios: formulas, input values, reporting period/basis, evidence, counter-evidence, missing data, and concise rationale. Calculate ratios deterministically in Python. Do not request or expose hidden private reasoning; retain an auditable rationale instead.

Use scenario branching for uncertain multi-year outcomes: bull/base/bear assumptions, report evidence, ratio implications, risks, monitorables, and pruning criteria. Selection: structured ratio analysis for fundamental health; scenario branching for the three-to-five-year rating. Self-consistency detects instability but cannot repair weak evidence. Escalate split votes or low confidence.

## 10. Multi-turn analyst conversation (six-plus turns)

In [ ]:
company_context={
 "company":"Reliance Industries Limited","horizon":"3–5 years; default 5 years",
 "evidence_status":"FY2025 segment overview partially loaded; full page-level extracts and actual prices still required",
 "price_data":"synthetic demo only","candidate_rating":"HOLD (provisional; do not publish)",
 "assumptions":["consumer businesses may grow","O2C remains cyclical","capex returns require monitoring"],
 "open_questions":["verified net debt trend?","audited consolidated ratios?","actual price/volume statistics?"]
}
turns=[
 ("Analyst","Why not BUY?"),
 ("Copilot","A BUY is not supported by this incomplete evidence pack; real prices and audited ratios are missing."),
 ("Analyst","Retail and digital are growing. Isn't that positive?"),
 ("Copilot","Yes. The official overview reports FY2025 revenue growth of 7.9% for Retail and 15.9% for Digital Services. This must be weighed against margins, cash conversion, debt, capex returns, and valuation."),
 ("Analyst","O2C EBITDA fell. Should this be SELL?"),
 ("Copilot","O2C revenue grew 11.0% while EBITDA growth was -11.9% in the official overview: a mixed signal, not sufficient alone for SELL."),
 ("Analyst","What would upgrade the view?"),
 ("Copilot","Verified actual prices, audited ratios, sustained margin improvement, cash conversion, manageable leverage, and capex milestones."),
 ("Analyst","Can we publish now?"),
 ("Copilot","No. Synthetic prices and missing audited ratios block publication; analyst review is required.")
]
for i,(speaker,msg) in enumerate(turns,1): print(f"Turn {i} | {speaker}: {msg}")

In [ ]:
if langchain_available:
    analyst_template=ChatPromptTemplate.from_messages([
      ("system","You are an educational RIL research copilot. Persistent context: {company_context}. Current rating: {rating}. State facts, assumptions, gaps, and review gate. Include: "+COMPLIANCE),
      ("human","Analyst challenge: {analyst_message}")
    ])
    print("Dynamic multi-turn template ready.")

## 11. Jinja2 research note, f-string alerts, and custom prompt

In [ ]:
note_template=Template('''# Reliance Industries — Long-Term Research Note
**Rating:** {{ rating }}
**Conviction:** {{ conviction }}
**Horizon:** {{ horizon }}

## Rationale
{{ rationale }}

## Risks
{% for x in risks %}- {{ x }}
{% endfor %}
## Monitorable indicators
{% for x in monitorables %}- {{ x }}
{% endfor %}
## Evidence gaps
{{ gaps }}

**Compliance disclaimer:** {{ disclaimer }}
''')
note=note_template.render(
 rating="HOLD (provisional / not publishable)",conviction="Low",horizon="3–5 years",
 rationale="Selected FY2025 segment facts show Retail and Digital Services growth, while O2C EBITDA declined. Full audited ratios, risk/debt/capex extracts, valuation context and real price history are required.",
 risks=["O2C margin cyclicality","Capex execution and returns","Competition and regulation","Unverified leverage and valuation"],
 monitorables=["Retail and Digital Services revenue/EBITDA","O2C EBITDA and margins","Operating cash flow and net debt","Capex milestones and return on capital","Actual adjusted price drawdown and volatility"],
 gaps="Historical price file is synthetic; ratio inputs and page-level citations are pending. Analyst review required.",
 disclaimer=COMPLIANCE)
print(note)
earnings_alert=f"RIL earnings alert: review revenue, EBITDA, segment margins, cash flow, debt and capex versus prior period and consensus. {COMPLIANCE}"
price_alert=f"RIL price alert: threshold crossed; verify actual adjusted prices and benchmark before attributing a cause. {COMPLIANCE}"
print(earnings_alert); print(price_alert)
custom_template=Template('''Instruction: {{ instruction }}
Context: {{ context }}
Financial-ratio glossary:
{% for name,definition in glossary.items() %}- {{ name }}: {{ definition }}
{% endfor %}
Input data: {{ input_data }}
Output indicator: {{ output_indicator }}
Compliance: {{ disclaimer }}''')
print(custom_template.render(instruction="Assess supplied ratios",context="RIL educational research",
 glossary=glossary,input_data="Insert audited values",output_indicator="Flag, evidence, missing inputs, confidence, review gate",disclaimer=COMPLIANCE))

## 12. LLM settings findings table

In [ ]:
settings=pd.DataFrame([
 ["Extraction","0.0–0.2","0.8–1.0","Low randomness; verify every figure against source."],
 ["Classification","0.0–0.2","0.8–1.0","Prioritise stable labels; test ambiguous cases."],
 ["Financial reasoning","0.0–0.2","0.8–1.0","Structured output and repeated stability checks; analyst review."],
 ["Narrative note","0.2–0.5","0.8–0.95","More variation for readability, but facts remain evidence-bound."]
],columns=["task","temperature","top_p","finding"])
settings

## 13. Technique-selection memo

- **Prompt iteration:** scope → structure → evidence citations and missing-data markers.
- **Zero-shot:** establishes a baseline and surfaces failure modes.
- **Few-shot:** clarifies mixed-signal and ambiguity labels.
- **Python ratios:** deterministic formulas and missing-input checks.
- **Self-consistency:** detects instability; split/low-confidence votes escalate.
- **Scenario branching:** suitable for long-horizon uncertainty and competing drivers.
- **LangChain templates:** separate persistent company context from analyst turns.
- **Jinja2/f-strings:** reproducible research notes and alerts with automatic disclaimer.
- **Human review gate:** blocks publication if source evidence, real prices, or validated ratios are absent.

## 14. Held-out evaluation and limitations

In [ ]:
train,test=train_test_split(events,test_size=0.2,random_state=42,stratify=events.event_type)
p=test.headline_or_snippet.apply(zero_shot)
type_acc=accuracy_score(test.event_type,[x[0] for x in p])
sent_acc=accuracy_score(test.sentiment,[x[1] for x in p])
{"held_out_event_type_accuracy":round(type_acc,3),
 "held_out_sentiment_accuracy":round(sent_acc,3),
 "required_target":0.80,
 "event_type_target_met":bool(type_acc>=0.80),
 "sentiment_target_met":bool(sent_acc>=0.80)}

### Final limitations and next steps
1. Replace the synthetic price CSV with the actual historical dataset specified in the assignment.
2. Add the full annual report and page-level extracts for Chairman's statement, segment performance, risk factors, and debt/capex.
3. Calculate the five ratios from audited values on one consistent reporting basis.
4. Keep a genuinely held-out manually reviewed test set and target at least 80% accuracy.
5. If using an API, log model/version, prompt version, temperature, top-p, output, parsing failures, and analyst decisions.
6. Do not publish a rating until evidence, valuation, and analyst approval are complete.

**Final disclaimer:** This is an educational prompt-engineering project, not investment advice or a recommendation to transact in any security.